<a href="https://colab.research.google.com/github/MatthewTsan/RAG-Toy/blob/start/Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install sentence-transformers faiss-cpu beautifulsoup4 html2text tqdm


In [2]:
# === pick one of: "dataset1", "dataset2", "dataset3" ===
DATASET = "dataset1"  # <- change here

if DATASET == "dataset1":
    with open("dataset1.jsonl","w") as f:
        f.write('{"id":"solar_system","url":"https://simple.wikipedia.org/wiki/Solar_System","title":"Solar System","text":""}\n')
elif DATASET == "dataset2":
    with open("dataset2.jsonl","w") as f:
        f.write('{"id":"sun","url":"https://simple.wikipedia.org/wiki/Sun","title":"Sun","text":""}\n')
        f.write('{"id":"earth","url":"https://simple.wikipedia.org/wiki/Earth","title":"Earth","text":""}\n')
        f.write('{"id":"moon","url":"https://simple.wikipedia.org/wiki/Moon","title":"Moon","text":""}\n')
        f.write('{"id":"mars","url":"https://simple.wikipedia.org/wiki/Mars","title":"Mars","text":""}\n')
        f.write('{"id":"jupiter","url":"https://simple.wikipedia.org/wiki/Jupiter","title":"Jupiter","text":""}\n')
elif DATASET == "dataset3":
    with open("dataset3.jsonl","w") as f:
        f.write('{"id":"apollo11","url":"https://en.wikipedia.org/wiki/Apollo_11","title":"Apollo 11","text":""}\n')
        f.write('{"id":"armstrong","url":"https://en.wikipedia.org/wiki/Neil_Armstrong","title":"Neil Armstrong","text":""}\n')
        f.write('{"id":"aldrin","url":"https://en.wikipedia.org/wiki/Buzz_Aldrin","title":"Buzz Aldrin","text":""}\n')
        f.write('{"id":"saturnv","url":"https://en.wikipedia.org/wiki/Saturn_V","title":"Saturn V","text":""}\n')


In [3]:
import json, re, os, time, hashlib, requests
from bs4 import BeautifulSoup
import html2text
from tqdm import tqdm

def read_jsonl(path):
    docs = []
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                docs.append(json.loads(line))
    return docs

def fetch_url(url, retries=3, sleep=1.5):
    headers = {"User-Agent": "Colab-RAG/1.0 (+https://colab.google.com/)"}
    for i in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=20)
            r.raise_for_status()
            return r.text
        except Exception as e:
            if i == retries - 1:
                raise
            time.sleep(sleep)

def html_to_markdown_text(html):
    # Remove navbars/footers/tables/refs roughly via BS4 first
    soup = BeautifulSoup(html, "html.parser")

    # Drop scripts/styles
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    # Wikipedia: remove tables, infoboxes, reference lists for simplicity
    for tag in soup.find_all(["table", "sup", "style", "img"]):
        tag.decompose()

    # Convert to markdown-like plain text
    h = html2text.HTML2Text()
    h.ignore_links = True
    h.ignore_images = True
    h.ignore_emphasis = False
    h.body_width = 0
    text = h.handle(str(soup))

    # Clean brackets like [1], [2], extra whitespace
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

def chunk_text(text, max_chars=1200, overlap=200):
    """
    Simple char-based splitter: robust enough for a start.
    """
    text = text.strip()
    if len(text) <= max_chars:
        return [text]

    chunks, start = [], 0
    while start < len(text):
        end = min(len(text), start + max_chars)
        # try to break at last sentence end before 'end'
        split_idx = text.rfind("\n", start, end)
        if split_idx == -1 or end - split_idx > max_chars * 0.6:
            split_idx = end
        chunks.append(text[start:split_idx].strip())
        if split_idx == len(text):
            break
        start = max(0, split_idx - overlap)
    return [c for c in chunks if c]


In [4]:
DATASET_PATH = {
    "dataset1": "dataset1.jsonl",
    "dataset2": "dataset2.jsonl",
    "dataset3": "dataset3.jsonl",
}[DATASET]

raw_docs = read_jsonl(DATASET_PATH)
len(raw_docs), raw_docs[:2]

(1,
 [{'id': 'solar_system',
   'url': 'https://simple.wikipedia.org/wiki/Solar_System',
   'title': 'Solar System',
   'text': ''}])

In [5]:
ingested = []  # [{doc_id, url, title, chunk_id, text}]
for d in tqdm(raw_docs, desc="Ingesting"):
    url = d["url"]
    html = fetch_url(url)
    print("fetch html success")
    txt = html_to_markdown_text(html)
    print("html to markdown success")
    chunks = chunk_text(txt, max_chars=1200, overlap=180)
    print("chunk success")
    for i, ch in enumerate(chunks):
        ingested.append({
            "doc_id": d["id"],
            "url": url,
            "title": d.get("title", d["id"]),
            "chunk_id": i,
            "text": ch
        })

print(f"Docs: {len(raw_docs)}  |  Chunks: {len(ingested)}")


Ingesting:   0%|          | 0/1 [00:00<?, ?it/s]

fetch html success
html to markdown success


Ingesting:   0%|          | 0/1 [00:13<?, ?it/s]


KeyboardInterrupt: 